In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [ ]:
os.getcwd()

In [ ]:
os.chdir("/home/dermot.kelly/Dermot_analysis/Phd/Paper_2/rumen_microbiome_pipeline/")

In [ ]:
FEATURE_FILE = "full_combined_results/rumen_combined_feature-table.tsv"
TAX_FILE = (
    "corrected_taxonomy_515F_806R/"
    "taxonomy_export/taxonomy.tsv"
)

OUTDIR = Path("corrected_taxonomy_515F_806R")
OUTDIR.mkdir(exist_ok=True)

In [ ]:
tax = pd.read_csv(TAX_FILE, sep="\t")
tax.head()

In [ ]:
def parse_taxonomy(t):
    ranks = {"Kingdom":"Unassigned","Phylum":"Unassigned","Class":"Unassigned",
             "Order":"Unassigned","Family":"Unassigned","Genus":"Unassigned","Species":"Unassigned"}

    if pd.isna(t):
        return pd.Series(ranks)

    parts = [x.strip() for x in str(t).split(";")]

    rank_map = {
        "d__":"Kingdom",
        "p__":"Phylum",
        "c__":"Class",
        "o__":"Order",
        "f__":"Family",
        "g__":"Genus",
        "s__":"Species"
    }

    for p in parts:
        for prefix, rank in rank_map.items():
            if p.startswith(prefix):
                val = p.replace(prefix, "").strip()
                if val == "":
                    val = "Unassigned"
                ranks[rank] = val

    return pd.Series(ranks)

parsed = tax["Taxon"].apply(parse_taxonomy)
tax = pd.concat([tax, parsed], axis=1)

tax.head()

In [ ]:
for rank in ["Phylum","Class","Order","Family","Genus","Species"]:
    assigned = (tax[rank] != "Unassigned").sum()
    pct = assigned / len(tax) * 100
    print(rank, assigned, round(pct,1))

In [ ]:
feat = pd.read_csv(
    FEATURE_FILE,
    sep="\t",
    skiprows=1
)

feat.rename(columns={feat.columns[0]: "FeatureID"}, inplace=True)
feat.head()

In [ ]:
feat = feat.set_index("FeatureID")

sample_matrix = feat.T
sample_matrix.index.name = "SampleID"
sample_matrix.reset_index(inplace=True)

sample_matrix.head()

In [ ]:
# ---- genus lookup ----
lookup = tax[["Feature ID","Genus"]].copy()
lookup.columns = ["FeatureID","Genus"]
lookup["Genus"] = lookup["Genus"].replace("", "Unassigned")

# ---- collapse ASVs to genus (fast column-grouping, no giant melt) ----
# feat is features x samples, indexed by FeatureID
feat_genus = feat.copy()
feat_genus.index = feat_genus.index.map(
    lookup.set_index("FeatureID")["Genus"]
).fillna("Unassigned")

genus_by_sample = feat_genus.groupby(level=0).sum()      # rows = genus, cols = samples
genus_wide = genus_by_sample.T.reset_index().rename(columns={"index": "SampleID"})

genus_wide.head()

In [ ]:
print(genus_wide.shape)          
print(genus_wide["SampleID"].nunique())   # expect 1487, all unique
# genus matrix row sums should equal the original per-sample depth
genus_wide.set_index("SampleID").sum(axis=1).describe()

In [ ]:
# what are the lowest-depth samples?
depths = genus_wide.set_index("SampleID").sum(axis=1).sort_values()
depths.head(15)

In [ ]:
print("samples < 1000 reads:", (depths < 1000).sum())
print("samples < 5000 reads:", (depths < 5000).sum())

In [ ]:
import re

# corrected: allow optional underscore before trailing number (handles ControlP_ive_5)
control_pattern = r'__(?:ControlN_ive|ControlNive|ControlP_ive|Minus|Plus|N|P)_?\d*$'
is_control = genus_wide["SampleID"].str.contains(control_pattern, regex=True)

print("Flagged as control/blank:", is_control.sum())      # expect 28
print(sorted(genus_wide.loc[is_control, "SampleID"].tolist()))

In [ ]:
# Save corrected genus-level count tables
genus_wide.to_csv(
    OUTDIR / "genus_counts_with_controls_515F_806R.csv",
    index=False
)

genus_bio = genus_wide[~is_control].reset_index(drop=True)

genus_bio.to_csv(
    OUTDIR / "genus_counts_no_controls_515F_806R.csv",
    index=False
)

print(f"with_controls: {genus_wide.shape[0]}")   # expect 1487
print(f"no_controls:   {genus_bio.shape[0]}")    # expect 1459

# Save corrected taxonomy lookup
tax.to_csv(
    OUTDIR / "taxonomy_lookup_515F_806R.csv",
    index=False
)